# ResumeFit AI — End-to-End LLM Fine-Tuning with QLoRA / PEFT
## Fine-Tuning Qwen 2.5 on Domain Resume-Job Alignment Data

This notebook demonstrates the complete LLM fine-tuning lifecycle for **ResumeFit AI**:
1. **Environment Setup & GPU Check** (NVIDIA T4 / A10G / V100 / A100)
2. **Dataset Loading & Preprocessing** (Structured Resume-JD alignment data)
3. **Pretrained Base Model & Tokenizer Loading** (`Qwen/Qwen2.5-1.5B-Instruct`)
4. **4-Bit Quantization (QLoRA) & BitsAndBytes Configuration**
5. **Parameter-Efficient Fine-Tuning (PEFT) with LoRA Adapters**
6. **Supervised Fine-Tuning (SFT) Loop with TRL / Transformers**
7. **Training Loss Monitoring & Evaluation Curves**
8. **LoRA Adapter Weight Export & Verification**
9. **Production Inference: Structured Resume Analysis**
10. **Benchmark Comparison vs. Untuned Baseline**

### Stage 1: Hardware Verification & Package Installation

In [ ]:
!nvidia-smi
!pip install -q torch transformers datasets peft trl bitsandbytes accelerate safetensors matplotlib

In [ ]:
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

### Stage 2: Load ResumeFit AI Training Dataset

In [ ]:
import json
from datasets import Dataset

# Sample structured training example
sample_data = [
    {
        "prompt": "<|im_start|>system\nYou are ResumeFit AI, an expert resume coach. Provide a valid JSON analysis.\n<|im_end|>\n<|im_start|>user\n## RESUME:\nAlex Rivera, ML Engineer with 5 years experience in PyTorch, transformers, FastAPI, Docker.\n\n## JOB DESCRIPTION:\nSenior ML Engineer with experience in PyTorch, PEFT/LoRA, vLLM, and cloud deployment.\n\n## TARGET ROLE:\nSenior Machine Learning Engineer<|im_end|>\n<|im_start|>assistant\n",
        "completion": json.dumps({
            "overall_match": 88,
            "technical_skill_match": 90,
            "matching_skills": ["Python", "PyTorch", "FastAPI", "Docker"],
            "missing_skills": ["vLLM", "PEFT/LoRA"],
            "summary_recommendation": "Strong alignment. Highlight model quantization and serving latency metrics to meet senior requirements.",
            "bullet_improvements": [
                {
                    "original": "Worked on ML models using PyTorch.",
                    "improved": "Engineered distributed PyTorch LLM training pipelines, cutting inference latency by 45% using vLLM.",
                    "reason": "Adds quantifiable metrics and specific serving framework."
                }
            ],
            "hallucination_check": "PASS"
        }, indent=2) + "<|im_end|>"
    }
] * 100

dataset = Dataset.from_list(sample_data)
print(f"Dataset prepared with {len(dataset)} examples.")

### Stage 3: Load Pretrained Base Model with 4-bit QLoRA Quantization

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# 4-Bit NormalFloat Quantization for QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config if torch.cuda.is_available() else None,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
print("Base model loaded successfully.")

### Stage 4: Attach PEFT / LoRA Adapters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if torch.cuda.is_available():
    model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()

### Stage 5: Execute Supervised Fine-Tuning (SFT) Training Loop

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./resumefit_lora_checkpoint",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    optim="paged_adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
    report_to="none",
)

# Format inputs
def tokenize_func(examples):
    texts = [p + c for p, c in zip(examples["prompt"], examples["completion"])]
    enc = tokenizer(texts, truncation=True, max_length=1024, padding="max_length")
    enc["labels"] = enc["input_ids"].copy()
    return enc

tokenized_ds = dataset.map(tokenize_func, batched=True)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_ds,
)

print("Starting LoRA training...")
trainer.train()

### Stage 6: Export LoRA Adapter Weights & Save Artifacts

In [ ]:
output_dir = "./models/adapters/resumefit-qwen2.5-lora"
peft_model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"LoRA adapter weights and tokenizer saved successfully to: {output_dir}")

import os
print("Exported files:", os.listdir(output_dir))

### Stage 7: Inference with Trained LoRA Adapter

In [ ]:
from peft import PeftModel

# Load base model + LoRA adapter for production inference
inference_model = PeftModel.from_pretrained(model, output_dir)
inference_model.eval()

test_prompt = "<|im_start|>system\nYou are ResumeFit AI, an expert resume coach. Output JSON.\n<|im_end|>\n<|im_start|>user\n## RESUME:\nExperienced Software Developer proficient in Python, FastAPI, and Docker.\n\n## JOB DESCRIPTION:\nBackend Developer with Python, FastAPI, Microservices, and Kubernetes.\n\n## TARGET ROLE:\nBackend Developer<|im_end|>\n<|im_start|>assistant\n"

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = inference_model.generate(**inputs, max_new_tokens=512, temperature=0.1)

result_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Model Output:\n", result_text)